# 数独门控实验 · 从三个种子补到十个

本笔记本补训种子 59、71、83、97、109、127、139。每个种子分别训练全通道门控与分区门控，各完成 5,000 次 AdamW 更新。相同种子的两个模型使用相同的主干初值、批次和数字置换。按种子交替两种门控的训练顺序。种子 17、29、43 的六个现有模型及逐格轨迹从上轮归档读取。

训练使用上轮的 2,048 道题，展开 8 轮。评估沿用相同顺序的 512 道原分布新题和 512 道困难题，原始空格的预测数字逐轮从 1 至 32 全部保存。初始化时校验训练集和两套评估题的 SHA-256。

主要指标沿用上轮定义。在同一道难题中，第 8 轮两个模型都预测正确的格子，统计其在第 9–32 轮是否曾改错。第 8 轮两个模型都预测错误的格子，统计第 32 轮是否改对。每个种子内逐格配对，再按种子等权汇总。全部十个种子和新增七个种子分别报告。纠错率的非劣效幅度沿用上轮设置，未事先指定。

点击“全部运行”，输入资源面板显示的当前 CU/小时。每次训练完成，下一单元将继续运行；权重和两套题的逐格预测同时保存在当次输出。最后一个单元保存完成清单、估算本轮 CU 并释放 GPU。十种子的综合统计由这些输出与原归档合并计算。


In [1]:
import time
if 'STRESS_SESSION_START' not in globals():
    STRESS_SESSION_START=time.monotonic()


In [2]:
# 可离线生成的 9×9 数独。导出为 notebook 单元后只依赖 Python 和 NumPy。
import hashlib
import random
import numpy as np

_SDK_ALL = (1 << 9) - 1
_SDK_ROW = tuple(i // 9 for i in range(81))
_SDK_COL = tuple(i % 9 for i in range(81))
_SDK_BOX = tuple(i // 27 * 3 + i % 9 // 3 for i in range(81))


def _sdk_completion(puzzle, *, known=None, rng=None):
    """Return one completion, excluding known when supplied. Internal grids are valid."""
    board = list(puzzle)
    rows, cols, boxes = [0] * 9, [0] * 9, [0] * 9
    empty = []
    for i, digit in enumerate(board):
        if digit:
            bit = 1 << (digit - 1)
            rows[_SDK_ROW[i]] |= bit
            cols[_SDK_COL[i]] |= bit
            boxes[_SDK_BOX[i]] |= bit
        else:
            empty.append(i)
    if rng is not None:
        rng.shuffle(empty)

    def search(depth, differs):
        if depth == len(empty):
            return board.copy() if known is None or differs else None
        best, best_mask, best_n = depth, 0, 10
        for j in range(depth, len(empty)):
            i = empty[j]
            mask = _SDK_ALL & ~(rows[_SDK_ROW[i]] | cols[_SDK_COL[i]] | boxes[_SDK_BOX[i]])
            count = mask.bit_count()
            if count < best_n:
                if count == 0:
                    return None
                best, best_mask, best_n = j, mask, count
                if count == 1:
                    break
        empty[depth], empty[best] = empty[best], empty[depth]
        i = empty[depth]
        r, c, b = _SDK_ROW[i], _SDK_COL[i], _SDK_BOX[i]
        candidates = []
        while best_mask:
            bit = best_mask & -best_mask
            candidates.append(bit)
            best_mask ^= bit
        if best_n > 1:
            if rng is not None:
                rng.shuffle(candidates)
            elif known is not None and not differs:
                preferred_last = 1 << (known[i] - 1)
                if preferred_last in candidates:
                    candidates.remove(preferred_last)
                    candidates.append(preferred_last)
        for bit in candidates:
            digit = bit.bit_length()
            board[i] = digit
            rows[r] |= bit
            cols[c] |= bit
            boxes[b] |= bit
            result = search(depth + 1, differs or (known is not None and digit != known[i]))
            if result is not None:
                return result
            rows[r] ^= bit
            cols[c] ^= bit
            boxes[b] ^= bit
        board[i] = 0
        empty[depth], empty[best] = empty[best], empty[depth]
        return None

    return search(0, False)


def _sdk_split_id(solution):
    # The cycle lengths of the digit permutation formed by two rows survive
    # digit relabeling and column permutation. Sorting both orientations also
    # makes the signature invariant to row order and transposition. Thus every
    # board in a Sudoku symmetry orbit is assigned to the same split. Distinct
    # orbits can share a signature, which only keeps extra examples together.
    rows = [solution[i : i + 9] for i in range(0, 81, 9)]

    def fingerprint(lines):
        result = []
        for i, left in enumerate(lines):
            for right in lines[i + 1 :]:
                perm = [0] * 10
                for a, b in zip(left, right):
                    perm[a] = b
                seen, lengths = 0, []
                for digit in range(1, 10):
                    length = 0
                    while not (seen & (1 << digit)):
                        seen |= 1 << digit
                        digit = perm[digit]
                        length += 1
                    if length:
                        lengths.append(length)
                result.append(tuple(sorted(lengths)))
        return tuple(sorted(result))

    signature = sorted((fingerprint(rows), fingerprint(list(zip(*rows)))))
    digest = hashlib.blake2b(repr(signature).encode("ascii"), digest_size=8).digest()
    bucket = int.from_bytes(digest, "little") % 10
    return "val" if bucket == 8 else "test" if bucket == 9 else "train"


def _sdk_puzzle(solution, rng, index):
    # Mix starting clue densities. Clues added to force uniqueness count too.
    low, high = ((24, 31), (32, 40), (43, 52))[index % 3]
    puzzle = [0] * 81
    for i in rng.sample(range(81), rng.randint(low, high)):
        puzzle[i] = solution[i]
    while True:
        alternative = _sdk_completion(puzzle, known=solution)
        if alternative is None:
            return puzzle
        i = rng.choice([i for i in range(81) if alternative[i] != solution[i]])
        puzzle[i] = solution[i]


def make_sudoku_splits(n_train=2048, n_val=256, n_test=256, seed=2026):
    """Return {train, val, test: (puzzles[N,81], solutions[N,81])}, uint8."""
    sizes = dict(train=n_train, val=n_val, test=n_test)
    if any(not isinstance(n, (int, np.integer)) or n < 0 for n in sizes.values()):
        raise ValueError("Split sizes must be nonnegative integers.")
    rng = random.Random(int(seed))
    splits = {k: (np.empty((n, 81), np.uint8), np.empty((n, 81), np.uint8))
              for k, n in sizes.items()}
    counts = dict.fromkeys(sizes, 0)
    seen = set()
    while counts != sizes:
        solution = _sdk_completion([0] * 81, rng=rng)
        puzzle_seed = rng.getrandbits(64)
        key = _sdk_split_id(solution)
        i = counts[key]
        code = bytes(solution)
        if i == sizes[key] or code in seen:
            continue
        seen.add(code)
        puzzle = _sdk_puzzle(solution, random.Random(puzzle_seed), i)
        splits[key][0][i] = puzzle
        splits[key][1][i] = solution
        counts[key] += 1
    return splits


def sample_sudoku_batch(split, batch_size, rng, augment=True):
    """Sample with replacement. Pass a seeded np.random.Generator. No split mutation."""
    puzzles, solutions = split
    picks = rng.integers(0, len(puzzles), size=batch_size)
    puzzles, solutions = puzzles[picks], solutions[picks]
    if not augment:
        return puzzles, solutions

    def shuffled_axis():
        return np.concatenate([3 * band + rng.permutation(3) for band in rng.permutation(3)])

    grid = np.arange(81).reshape(9, 9)
    if rng.integers(2):
        grid = grid.T
    positions = grid[np.ix_(shuffled_axis(), shuffled_axis())].reshape(-1)
    digit_map = np.zeros((batch_size, 10), np.uint8)
    digit_map[:, 1:] = 1 + np.argsort(rng.random((batch_size, 9)), axis=1)
    return tuple(np.take_along_axis(digit_map, values[:, positions], axis=1)
                 for values in (puzzles, solutions))


In [3]:
import torch
from torch import nn
from torch.nn import functional as F


class SudokuBlock(nn.Module):
    """One shared pre-norm Transformer block with bidirectional attention."""

    def __init__(self, width, heads, ffn_width):
        super().__init__()
        self.heads = heads
        self.head_width = width // heads
        self.attn_norm = nn.LayerNorm(width)
        self.qkv = nn.Linear(width, 3 * width)
        self.attn_out = nn.Linear(width, width)
        positions = torch.arange(81)
        rows, cols = positions // 9, positions % 9
        boxes = (rows // 3) * 3 + cols // 3
        same_row = rows[:, None] == rows[None, :]
        same_col = cols[:, None] == cols[None, :]
        same_box = boxes[:, None] == boxes[None, :]
        global_attention = torch.ones_like(same_row)
        head_types = (same_row, same_col, same_box, global_attention)
        masks = [head_types[i % 4] for i in range(heads)]
        if heads < 4:
            masks[-1] = global_attention
        # SDPA True entries permit attention. Shared by all variants, this
        # puzzle-topology prior contains only locations, never puzzle answers.
        self.register_buffer("attention_mask", torch.stack(masks)[None], persistent=False)
        self.ffn_norm = nn.LayerNorm(width)
        self.ffn = nn.Sequential(
            nn.Linear(width, ffn_width), nn.GELU(), nn.Linear(ffn_width, width)
        )

    def forward(self, state):
        batch, cells, width = state.shape
        qkv = self.qkv(self.attn_norm(state))
        qkv = qkv.reshape(batch, cells, 3, self.heads, self.head_width)
        query, key, value = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        context = F.scaled_dot_product_attention(
            query, key, value, attn_mask=self.attention_mask, is_causal=False
        )
        context = context.transpose(1, 2).reshape(batch, cells, width)
        proposal = state + self.attn_out(context)
        return proposal + self.ffn(self.ffn_norm(proposal))


class SudokuLoopTransformer(nn.Module):
    """Inputs are digits 0–9 (0 = blank). Logit indices 0–8 denote digits 1–9."""

    def __init__(self, variant, width=96, heads=4, ffn_width=192, core_width=32):
        super().__init__()
        if variant not in ("plain", "full_gate", "split_gate"):
            raise ValueError(f"Unknown model variant: {variant}")
        if heads < 1 or width % heads or not 0 < core_width < width:
            raise ValueError("Use a width divisible by heads and 0 < core_width < width.")
        self.variant = variant
        self.digit = nn.Embedding(10, width)
        self.row = nn.Embedding(9, width)
        self.col = nn.Embedding(9, width)
        self.box = nn.Embedding(9, width)
        cell = torch.arange(81)
        rows, cols = cell // 9, cell % 9
        self.register_buffer("rows", rows, persistent=False)
        self.register_buffer("cols", cols, persistent=False)
        self.register_buffer("boxes", (rows // 3) * 3 + cols // 3, persistent=False)
        self.block = SudokuBlock(width, heads, ffn_width)
        self.readout = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 9))
        if variant != "plain":
            self.gate = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, core_width))
            # A full-stream gate shares one decision per group of channels. Both
            # controls have the same number of learned gates and the same GEMMs.
            channel = torch.arange(width)
            self.register_buffer("gate_group", channel % core_width, persistent=False)
            gated = channel < core_width if variant == "split_gate" else channel >= 0
            self.register_buffer("gated_channels", gated, persistent=False)

    def initial_state(self, puzzles):
        xy = self.row(self.rows) + self.col(self.cols) + self.box(self.boxes)
        return 0.5 * (self.digit(puzzles.long()) + xy)

    def step(self, state, gate_mode="learned"):
        proposal = self.block(state)
        if self.variant == "plain":
            return proposal
        gate = torch.sigmoid(self.gate(proposal)).to(dtype=state.dtype)
        if gate_mode == "open":
            gate = torch.ones_like(gate)
        elif gate_mode == "closed":
            gate = torch.zeros_like(gate)
        elif gate_mode != "learned":
            raise ValueError(f"Unknown gate intervention: {gate_mode}")
        gate = gate.index_select(-1, self.gate_group)
        # Only this interpolation can write the protected channels. Attention
        # and FFN proposals can read all channels. The readout never writes back.
        gated_proposal = torch.lerp(state, proposal, gate)
        return torch.where(self.gated_channels, gated_proposal, proposal)

    def forward(self, puzzles, steps, return_all=False, gate_mode="learned"):
        if puzzles.ndim != 2 or puzzles.shape[1] != 81 or steps < 1:
            raise ValueError("Expected [batch, 81] puzzles and steps >= 1.")
        state = self.initial_state(puzzles)
        if return_all:
            logits = []
            for _ in range(steps):
                state = self.step(state, gate_mode=gate_mode)
                logits.append(self.readout(state))
            return torch.stack(logits, dim=1)  # [batch, steps, 81, 9]
        for _ in range(steps):
            state = self.step(state, gate_mode=gate_mode)
        return self.readout(state)  # [batch, 81, 9]


In [4]:
import math
import time
from contextlib import nullcontext

import torch
from torch.nn import functional as F


FORMAL_VARIANTS = ("plain", "full_gate", "split_gate")
FORMAL_SEEDS = (17, 29, 43, 59, 71, 83, 97, 109, 127, 139)
FORMAL_DEPTHS = (8, 16, 32)


@torch.inference_mode()
def evaluate_boards(model, split, device, precision, batch_size=64, check_budget=None):
    """Each column is in source-puzzle order. Givens are copied when scoring a board."""
    model.eval()
    x_all, y_all = split
    if len(x_all) == 0 or len(x_all) != len(y_all):
        raise ValueError("Evaluation needs a nonempty, aligned split.")
    # Batch all CPU transfers per minibatch; keep counts as native Python ints.
    batches = []
    for start in range(0, len(x_all), batch_size):
        if check_budget:
            check_budget()
        x = torch.as_tensor(x_all[start:start + batch_size], device=device).long()
        y = torch.as_tensor(y_all[start:start + batch_size], device=device).long()
        blank = x.eq(0)
        cols = [blank.sum(1)]
        finite = []
        with precision():
            state = model.initial_state(x)
            for d in range(1, FORMAL_DEPTHS[-1] + 1):
                state = model.step(state)
                if d not in FORMAL_DEPTHS:
                    continue
                logits = model.readout(state)
                finite.append(torch.isfinite(logits).all())
                correct = logits.argmax(-1).add(1).eq(y) & blank
                if d == FORMAL_DEPTHS[0]:
                    right8 = correct
                right = correct.sum(1)
                ruined = (right8 & ~correct).sum(1)
                repaired = (correct & ~right8).sum(1)
                cols.extend((right, ruined, repaired))
        if not torch.stack(finite).all().item():
            raise FloatingPointError("Evaluation produced nonfinite logits.")
        batches.append(torch.stack(cols, dim=1).cpu())
    values = torch.cat(batches)
    n_blank = values[:, 0]
    at_eight = values[:, 1]
    result = {"n_blank": n_blank.tolist(), "right8": at_eight.tolist(), "by_depth": {}}
    for j, depth in enumerate(FORMAL_DEPTHS):
        offset = j * 3 + 1
        right, ruined, repaired = values[:, offset:offset + 3].unbind(1)
        if not torch.equal(right, at_eight - ruined + repaired):
            raise AssertionError("Per-board flip counts do not reconcile.")
        result["by_depth"][str(depth)] = {
            "exact": right.eq(n_blank).tolist(), "right": right.tolist(),
            "ruined": ruined.tolist(), "repaired": repaired.tolist(),
        }
    return result


def fit_formal_arm(variant, seed, splits, *, device="cuda", check_budget=None,
                   progress=None, total_steps=5000):
    """Train one fixed-recipe arm; the caller chooses its held-out readouts."""
    if variant not in FORMAL_VARIANTS or seed not in FORMAL_SEEDS or total_steps < 1:
        raise ValueError("Invalid formal variant, seed or step count.")
    device = torch.device(device)
    cuda = device.type == "cuda"
    if cuda:
        torch.cuda.set_device(device.index if device.index is not None else torch.cuda.current_device())
    torch.set_float32_matmul_precision("high")
    dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if cuda else torch.float32
    precision = (lambda: torch.autocast("cuda", dtype=dtype)) if cuda else nullcontext
    sync = (lambda: torch.cuda.synchronize(device)) if cuda else (lambda: None)
    torch.manual_seed(seed)
    model = SudokuLoopTransformer(variant, width=96, heads=4, ffn_width=192, core_width=32).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    scaled = cuda and dtype == torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=scaled, init_scale=256.0)
    batch_seed = seed + 1010
    gen = torch.Generator(device=device).manual_seed(batch_seed)
    train_x, train_y = [torch.as_tensor(a, device=device).long() for a in splits["train"]]
    if len(train_x) == 0 or train_x.shape != train_y.shape:
        raise ValueError("Training needs a nonempty, aligned split.")

    model.train()
    sync()
    started = time.perf_counter()
    overflow_retries = 0
    for step in range(1, total_steps + 1):
        if check_budget:
            check_budget()
        ids = torch.randint(len(train_x), (64,), device=device, generator=gen)
        x, y = train_x[ids], train_y[ids]
        digit_map = torch.rand((64, 9), device=device, generator=gen).argsort(1) + 1
        digit_map = torch.cat((torch.zeros_like(digit_map[:, :1]), digit_map), dim=1)
        x, y = digit_map.gather(1, x), digit_map.gather(1, y)
        target = (y - 1).masked_fill(x.ne(0), -100)
        # FP16 overflow retries reuse this very batch and never count as updates.
        for attempt in range(9):
            opt.zero_grad(set_to_none=True)
            with precision():
                logits = model(x, steps=8)
                loss = F.cross_entropy(logits.reshape(-1, 9), target.reshape(-1))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if (torch.isfinite(loss.detach()) & torch.isfinite(grad_norm)).item():
                scaler.step(opt)
                scaler.update()
                break
            if not scaled or attempt == 8:
                raise FloatingPointError(f"Nonfinite training value: {variant}, seed {seed}, update {step}")
            if check_budget:
                check_budget()
            scaler.update(new_scale=scaler.get_scale() / 2.0)
            overflow_retries += 1
        if progress and (step % 1000 == 0 or step == total_steps):
            progress(variant, seed, step, total_steps)
    sync()
    train_seconds = time.perf_counter() - started
    final_loss = float(loss.detach())
    if not math.isfinite(final_loss) or not all(torch.isfinite(p).all().item() for p in model.parameters()):
        raise FloatingPointError("Final trained parameters or loss are nonfinite.")
    if not all(int(opt.state[p]["step"].item()) == total_steps for p in model.parameters()):
        raise RuntimeError("Optimizer step counters do not match the completed updates.")
    record = {
        "variant": variant, "seed": seed, "batch_seed": batch_seed, "steps": total_steps,
        "parameters": sum(p.numel() for p in model.parameters()), "last_loss": final_loss,
        "precision": str(dtype), "torch_version": str(torch.__version__),
        "gpu": torch.cuda.get_device_name(device) if cuda else "CPU",
        "train_seconds": train_seconds, "updates_per_second": total_steps / train_seconds,
        "fp16_overflow_retries": overflow_retries, "splits": {}, "eval_split_seconds": {},
    }
    return model, record, device, precision, sync


def train_one(variant, seed, splits, *, device="cuda", save_run=None, check_budget=None,
              progress=None, total_steps=5000):
    """Train a fresh fixed-recipe arm. A completed record contains only JSON values."""
    model, record, device, precision, sync = fit_formal_arm(
        variant, seed, splits, device=device, check_budget=check_budget,
        progress=progress, total_steps=total_steps
    )
    for split_name in ("val", "test"):
        sync()
        before = time.perf_counter()
        record["splits"][split_name] = evaluate_boards(
            model, splits[split_name], device, precision, batch_size=64, check_budget=check_budget
        )
        sync()
        record["eval_split_seconds"][split_name] = time.perf_counter() - before
    record["eval_seconds"] = sum(record["eval_split_seconds"].values())
    if save_run:
        save_run(record)
    return record


In [5]:
import base64
import hashlib
import io
import os
from pathlib import Path
import tempfile
import time
import zlib

import numpy as np


_STRESS_CODEC = "u8-xor-rounds-blank-major-zlib-base64-v1"


def _trajectory_grids(split):
    puzzles, solutions = map(np.asarray, split)
    if (puzzles.ndim != 2 or puzzles.shape[1] != 81 or not len(puzzles)
            or puzzles.shape != solutions.shape):
        raise ValueError("Expected matching, nonempty [n,81] puzzle and solution arrays.")
    if (puzzles.dtype.kind not in "ui" or solutions.dtype.kind not in "ui"
            or puzzles.min() < 0 or puzzles.max() > 9
            or solutions.min() < 1 or solutions.max() > 9
            or np.any((puzzles != 0) & (puzzles != solutions))):
        raise ValueError("Invalid Sudoku digit values or conflicting givens.")
    return (np.ascontiguousarray(puzzles, dtype=np.uint8),
            np.ascontiguousarray(solutions, dtype=np.uint8))


def _trajectory_grid_hash(puzzles, solutions):
    # Hash version + board count (unsigned little-endian 64-bit) + C-order
    # puzzle bytes + C-order solution bytes. This binds the entire board order.
    digest = hashlib.sha256(b"sudoku-grid-order-v1\0" + len(puzzles).to_bytes(8, "little"))
    digest.update(memoryview(puzzles))
    digest.update(memoryview(solutions))
    return digest.hexdigest()


def pack_blank_predictions(prediction, split):
    """Losslessly encode [round, blank]. Blank order follows puzzles[puzzles==0]."""
    puzzles, solutions = _trajectory_grids(split)
    n_blanks = int(np.count_nonzero(puzzles == 0))
    values = np.asarray(prediction)
    if values.dtype.kind not in "ui" or values.shape != (32, n_blanks):
        raise ValueError("Expected 32 rows and one column per originally blank cell.")
    if n_blanks and (values.min() < 1 or values.max() > 9):
        raise ValueError("Predictions must use digits 1 through 9.")
    values = np.ascontiguousarray(values, dtype=np.uint8)
    # Store the first guess and XORs with the previous guess. Stable parts of a
    # cell's path then become runs of zeros even for sets wider than 32 KiB.
    delta = np.empty_like(values)
    delta[0] = values[0]
    np.bitwise_xor(values[1:], values[:-1], out=delta[1:])
    encoded = base64.b64encode(zlib.compress(delta.T.tobytes(order="C"), 9)).decode("ascii")
    return {
        "schema": _STRESS_CODEC, "shape": [32, n_blanks], "boards": len(puzzles),
        "split_sha256": _trajectory_grid_hash(puzzles, solutions),
        "raw_sha256": hashlib.sha256(memoryview(values)).hexdigest(), "data": encoded,
    }


def decode_blank_predictions(packed, split):
    """Restore all 32 argmax digits, checking source order, byte counts and digest."""
    puzzles, solutions = _trajectory_grids(split)
    n_blanks = int(np.count_nonzero(puzzles == 0))
    if (packed["schema"] != _STRESS_CODEC or packed["boards"] != len(puzzles)
            or packed["shape"] != [32, n_blanks]
            or packed["split_sha256"] != _trajectory_grid_hash(puzzles, solutions)):
        raise ValueError("Trajectory schema, shape or source split hash mismatch.")
    encoded = base64.b64decode(packed["data"], validate=True)
    delta = zlib.decompress(encoded)
    if len(delta) != 32 * n_blanks:
        raise ValueError("Incorrect decompressed trajectory length.")
    by_blank = np.frombuffer(delta, dtype=np.uint8).reshape(n_blanks, 32)
    pred = np.ascontiguousarray(np.bitwise_xor.accumulate(by_blank, axis=1).T)
    if hashlib.sha256(memoryview(pred)).hexdigest() != packed["raw_sha256"]:
        raise ValueError("Decoded trajectory SHA-256 mismatch.")
    if n_blanks and (pred.min() < 1 or pred.max() > 9):
        raise ValueError("Out-of-range digit in decoded trajectory.")
    return pred


def capture_blank_predictions(model, split, device, precision, batch_size=64, check_budget=None):
    import torch

    model.eval()
    puzzles, solutions = _trajectory_grids(split)
    prediction = np.empty((32, np.count_nonzero(puzzles == 0)), dtype=np.uint8)
    offset = 0
    with torch.inference_mode():
        for start in range(0, len(puzzles), batch_size):
            if check_budget:
                check_budget()
            chunk = puzzles[start:start + batch_size]
            x = torch.as_tensor(chunk, device=device).long()
            digits, finite = [], []
            with precision():
                state = model.initial_state(x)
                for _ in range(32):
                    state = model.step(state)
                    logits = model.readout(state)
                    finite.append(torch.isfinite(logits).all())
                    digits.append(logits.argmax(-1).add(1).to(torch.uint8))
            if not torch.stack(finite).all().item():
                raise FloatingPointError("Nonfinite trajectory logits.")
            blank_mask = chunk == 0
            end = offset + np.count_nonzero(blank_mask)
            prediction[:, offset:end] = torch.stack(digits).cpu().numpy()[:, blank_mask]
            offset = end
    if offset != prediction.shape[1]:
        raise RuntimeError("Trajectory blank count mismatch.")
    return pack_blank_predictions(prediction, (puzzles, solutions))


def _save_stress_weights(model, directory, variant, seed, steps):
    import torch

    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    final = directory / f"sudoku-{variant}-seed{seed}-steps{steps}.pt"
    weights = {k: p.detach().cpu() for k, p in model.state_dict().items()}
    stream = io.BytesIO()
    torch.save(weights, stream)
    payload = stream.getvalue()
    digest = hashlib.sha256(payload).hexdigest()
    with tempfile.NamedTemporaryFile(dir=directory, delete=False) as pending:
        temp = Path(pending.name)
    try:
        with temp.open("wb") as f:
            f.write(payload)
            f.flush()
            os.fsync(f.fileno())
        if hashlib.sha256(temp.read_bytes()).hexdigest() != digest:
            raise IOError("Model weight checkpoint verification failed.")
        os.replace(temp, final)
    finally:
        temp.unlink(missing_ok=True)
    return {"filename": final.name, "bytes": len(payload), "sha256": digest}


def train_one_stress(variant, seed, splits, *, steps=5000, device="cuda", weights_dir=None,
                     save_run=None, check_budget=None, progress=None):
    """Use the formal training kernel; evaluate every supplied split except 'train'."""
    held_out = [key for key in splits if key != "train"]
    if not held_out or "train" not in splits:
        raise ValueError("Supply the training split and at least one distinct evaluation split.")
    model, record, device, precision, sync = fit_formal_arm(
        variant, seed, splits, device=device, total_steps=steps,
        check_budget=check_budget, progress=progress
    )
    if weights_dir is not None:
        record["weights"] = _save_stress_weights(model, weights_dir, variant, seed, steps)
    for name in held_out:
        sync()
        before = time.perf_counter()
        record["splits"][name] = capture_blank_predictions(
            model, splits[name], device, precision, batch_size=64, check_budget=check_budget
        )
        sync()
        record["eval_split_seconds"][name] = time.perf_counter() - before
    record["eval_seconds"] = sum(record["eval_split_seconds"].values())
    if save_run:
        save_run(record)
    return record


In [6]:
import base64
import hashlib
import io
import json
import random

import numpy as np


_STRESS_SEED = 20261127
_STRESS_UNITS = ([tuple(9*r+c for c in range(9)) for r in range(9)] +
                 [tuple(9*r+c for r in range(9)) for c in range(9)])
_STRESS_UNITS += [
    tuple(9 * (3 * br + i) + 3 * bc + j for i in range(3) for j in range(3))
    for br in range(3) for bc in range(3)
]
_STRESS_CELL_UNITS = [
    (i//9, 9+i%9, 18+i//27*3+i%9//3) for i in range(81)
]


def _stress_signature(solution):
    """Same full 64-bit symmetry-invariant digest used by the formal split rule."""
    grid = [solution[i:i+9] for i in range(0, 81, 9)]

    def row_pairs(lines):
        fingerprints = []
        for i, left in enumerate(lines):
            for right in lines[i+1:]:
                permutation = dict(zip(left, right))
                unseen = set(range(1, 10))
                lengths = []
                while unseen:
                    x = next(iter(unseen))
                    length = 0
                    while x in unseen:
                        unseen.remove(x)
                        x = permutation[x]
                        length += 1
                    lengths.append(length)
                fingerprints.append(tuple(sorted(lengths)))
        return tuple(sorted(fingerprints))

    signature = sorted((row_pairs(grid), row_pairs(list(zip(*grid)))))
    return int.from_bytes(hashlib.blake2b(repr(signature).encode("ascii"), digest_size=8).digest(),
                          "little")


def _stress_singles(puzzle):
    """Apply naked and row/column/box hidden singles simultaneously until stalled."""
    board = list(map(int, puzzle))
    rounds = 0
    while 0 in board:
        occupied = [0] * 27
        for i, d in enumerate(board):
            if d:
                bit = 1 << (d - 1)
                for u in _STRESS_CELL_UNITS[i]:
                    occupied[u] |= bit
        candidates = [0] * 81
        seen, repeated = [0] * 27, [0] * 27
        for i, d in enumerate(board):
            if d:
                continue
            r, c, b = _STRESS_CELL_UNITS[i]
            mask = 511 & ~(occupied[r] | occupied[c] | occupied[b])
            if not mask:
                raise ValueError("Contradictory Sudoku clues")
            candidates[i] = mask
            for u in (r, c, b):
                repeated[u] |= seen[u] & mask
                seen[u] |= mask
        changed = False
        for i, mask in enumerate(candidates):
            if not mask:
                continue
            forced = mask if mask.bit_count() == 1 else 0
            for u in _STRESS_CELL_UNITS[i]:
                forced |= mask & seen[u] & ~repeated[u]
            if forced:
                if forced.bit_count() != 1:
                    raise ValueError("Inconsistent hidden singles")
                board[i] = forced.bit_length()
                changed = True
        if not changed:
            break
        rounds += 1
    return rounds, board.count(0)


def _stress_cover_count(puzzle):
    """Independent exact-cover verifier. Count solutions, stopping at two."""
    rows = []
    columns = [set() for _ in range(324)]
    for i, given in enumerate(puzzle):
        r, c = divmod(i, 9)
        box = 3*(r//3) + c//3
        for d in range(9):
            if given and given != d+1:
                continue
            covered = (i, 81+9*r+d, 162+9*c+d, 243+9*box+d)
            for j in covered:
                columns[j].add(len(rows))
            rows.append(covered)
    active = dict(enumerate(columns))

    def search():
        if not active:
            return 1
        col = min(active.values(), key=len)
        found = 0
        for option in tuple(col):
            removed = []
            for k in rows[option]:
                previous = active.pop(k)
                removed.append((k, previous))
                for other in previous:
                    for neighbor in rows[other]:
                        if neighbor in active:
                            active[neighbor].remove(other)
            found += search()
            for k, previous in reversed(removed):
                for other in previous:
                    for neighbor in rows[other]:
                        if neighbor in active:
                            active[neighbor].add(other)
                active[k] = previous
            if found >= 2:
                return 2
        return found

    return search()


def _stress_hash(x, y):
    return hashlib.sha256(x.tobytes() + y.tobytes()).hexdigest()


def _stress_profile(x, y, must_stall):
    histogram, stalled_histogram, rounds = {}, {}, []
    clues = np.count_nonzero(x, axis=1)
    for grid, solution, n in zip(x.tolist(), y.tolist(), clues.tolist()):
        if any(sorted(solution[i] for i in unit) != list(range(1,10)) for unit in _STRESS_UNITS):
            raise ValueError("Invalid solution grid")
        if any(a and a != b for a, b in zip(grid, solution)) or _stress_cover_count(grid) != 1:
            raise ValueError("Puzzle verification failed")
        r, stalled = _stress_singles(grid)
        if must_stall and stalled == 0:
            raise ValueError("Singles solved a hard puzzle")
        rounds.append(r)
        histogram[n] = histogram.get(n, 0) + 1
        stalled_histogram[stalled] = stalled_histogram.get(stalled, 0) + 1
    return dict(n_boards=len(x), data_sha256=_stress_hash(x,y),
                clues_min=int(clues.min()), clues_median=float(np.median(clues)),
                clues_mean=float(clues.mean()), clues_max=int(clues.max()),
                clues_histogram=histogram, singles_stalled_histogram=stalled_histogram,
                singles_solved=stalled_histogram.get(0,0),
                singles_rounds_median=float(np.median(rounds)), singles_rounds_max=max(rounds))


def generate_stress_corpus(n=512, seed=_STRESS_SEED):
    """Rebuild the two held-out suites from data_cells.py routines without model access."""
    previous = make_sudoku_splits(n_train=2048, n_val=256, n_test=256, seed=2026)
    prior_x, prior_y = previous["test"]
    expected = "aefcf3dae5d714772666dc2636744a19fe926d42600b4ce9a0de9f7eb0d8505c"
    if _stress_hash(prior_x, prior_y) != expected:
        raise ValueError("Training-era test set does not match its published fingerprint")
    excluded = {_stress_signature(row) for row in prior_y.tolist()}
    if any(x % 10 != 9 for x in excluded):
        raise AssertionError("Fingerprint disagrees with the formal split rule")
    corpus = {}
    report = {"seed": seed, "prior_test_hash": expected, "bucket": 9,
              "excluded_formal_test_signatures":len(excluded), "profiles": {}}
    for offset, suite in enumerate(("original", "hard")):
        rng = random.Random(seed+offset)
        boards, solutions = [], []
        attempts, accepted_solutions = 0, 0
        kept_bands = [0] * 3
        while len(boards) < n:
            solution = _sdk_completion([0]*81, rng=rng)
            puzzle_rng = random.Random(rng.getrandbits(64))
            sig = _stress_signature(solution)
            if sig % 10 != 9 or sig in excluded:
                continue
            accepted_solutions += 1
            if suite == "original":
                band = len(boards) % 3
                board = _sdk_puzzle(solution, puzzle_rng, band)
                attempts += 1
            else:
                # Keep a maximum of one puzzle per previously unseen solution signature.
                # Sample the same three clue bands with equal probability per mask.
                for _ in range(64):
                    band = puzzle_rng.randrange(3)
                    board = _sdk_puzzle(solution, puzzle_rng, band)
                    attempts += 1
                    if _stress_singles(board)[1]:
                        break
                else:
                    continue
            excluded.add(sig)
            kept_bands[band] += 1
            boards.append(board)
            solutions.append(solution)
        x, y = np.asarray(boards,np.uint8), np.asarray(solutions,np.uint8)
        corpus[suite] = (x, y)
        profile = _stress_profile(x, y, suite=="hard")
        profile["eligible_masks_attempted"] = attempts
        profile["eligible_solutions_attempted"] = accepted_solutions
        profile["accepted_initial_clue_bands"] = kept_bands
        report["profiles"][suite] = profile
    return corpus, report


def pack_stress_corpus(corpus, report):
    packed = io.BytesIO()
    arrays = {f"{suite}_{label}": arr for suite, pair in corpus.items()
              for label, arr in zip(("x","y"), pair)}
    encoded_report = json.dumps(report, sort_keys=True, separators=(",",":")).encode("utf-8")
    np.savez_compressed(packed, **arrays, report=np.frombuffer(encoded_report, dtype=np.uint8))
    return base64.b64encode(packed.getvalue()).decode("ascii")


def _read_stress_archive(packed):
    if packed is None:
        packed = globals().get("STRESS_DATA_B64")
    if not packed:
        raise ValueError("Missing embedded Sudoku corpus")
    return np.load(io.BytesIO(base64.b64decode(packed, validate=True)), allow_pickle=False)


def make_stress_sets(packed=None):
    with _read_stress_archive(packed) as archive:
        report = json.loads(archive["report"].tobytes())
        pairs = {}
        for suite in ("original", "hard"):
            x, y = archive[suite+"_x"], archive[suite+"_y"]
            if _stress_hash(x,y) != report["profiles"][suite]["data_sha256"]:
                raise ValueError(f"Corpus digest mismatch for {suite}")
            pairs[suite] = (x, y)
        return pairs


def stress_data_report(packed=None):
    with _read_stress_archive(packed) as archive:
        return json.loads(archive["report"].tobytes())


In [7]:
import math
import sys
import threading
import time


class BudgetStop(RuntimeError):
    """The experiment must stop and preserve its current results."""


def release_colab_runtime():
    """Release the hosted VM using Colab's public runtime-management function."""
    from google.colab import runtime

    try:
        runtime.unassign()
    except Exception:
        # Retry once in case the backend briefly lost the connection.
        time.sleep(2)
        runtime.unassign()


class RuntimeBudget:
    """Wall-clock guard using the rate shown by the active Colab runtime."""

    def __init__(self, cu_per_hour, already_used_cu=0.0, *, session_start,
                 limit_cu=8.0, max_hours=2.0, reserve_minutes=3.0):
        self.rate = float(cu_per_hour)
        self.prior_cu = float(already_used_cu)
        self.start = float(session_start)
        self.limit_cu = float(limit_cu)
        max_hours = float(max_hours)
        reserve_minutes = float(reserve_minutes)
        values = (self.rate, self.prior_cu, self.start, self.limit_cu,
                  max_hours, reserve_minutes)
        if not all(math.isfinite(value) for value in values):
            raise ValueError("CU 费率、预算和时间都必须是有限数值。")
        if self.rate <= 0 or self.prior_cu < 0:
            raise ValueError("请填写资源面板当前的正数 CU/小时和非负的已用 CU。")
        if not (0 < self.limit_cu <= 8 and 0 < max_hours <= 2 and reserve_minutes > 0):
            raise ValueError("本实验的运行预算上限为 8 CU、2 小时；收尾时间须为正数。")
        if not 0 <= self.start <= time.monotonic():
            raise ValueError("session_start 应取自当前内核首次运行时的 time.monotonic()。")
        available_cu = max(0.0, self.limit_cu - self.prior_cu)
        allowed_seconds = 3600 * min(max_hours, available_cu / self.rate)
        self.release_deadline = self.start + allowed_seconds
        self.work_deadline = self.release_deadline - 60 * reserve_minutes
        self._watchdog = None

    def remaining_seconds(self):
        """Seconds available for training or evaluation before wrap-up begins."""
        return max(0.0, self.work_deadline - time.monotonic())

    def check(self):
        if self.remaining_seconds() == 0:
            raise BudgetStop("已到预留的收尾时间，停止训练并保存当前结果。")

    def summary(self):
        elapsed = max(0.0, time.monotonic() - self.start)
        return {
            "runtime_cu_per_hour_entered": self.rate,
            "previous_sessions_cu_entered": self.prior_cu,
            "this_session_seconds": round(elapsed, 2),
            "estimated_total_cu": round(self.prior_cu + self.rate * elapsed / 3600, 4),
            "software_cu_limit": self.limit_cu,
            "remaining_work_seconds": round(self.remaining_seconds(), 2),
        }

    def arm_watchdog(self):
        """Request VM release at the ceiling even if a training call blocks."""
        if self._watchdog is not None:
            return

        def on_deadline():
            print("已到运行时限，正在释放 Colab 运行时。", file=sys.stderr, flush=True)
            try:
                release_colab_runtime()
            except Exception as exc:
                print(f"Colab 自动释放失败，请手动断开并删除运行时。{exc}",
                      file=sys.stderr, flush=True)

        seconds = max(0.0, self.release_deadline - time.monotonic())
        self._watchdog = threading.Timer(seconds, on_deadline)
        self._watchdog.daemon = True
        self._watchdog.start()

    def disarm_watchdog(self):
        if self._watchdog is not None:
            self._watchdog.cancel()
            self._watchdog = None


In [8]:
from pathlib import Path
import base64
import hashlib
import json
import re
import zlib


def publish_bytes(name, blob):
    """Keep a lossless copy in notebook outputs, independent of Colab VM storage."""
    from IPython.display import HTML, display
    if not re.fullmatch(r'[a-zA-Z0-9_.-]+', name):
        raise ValueError('Invalid archive member name')
    digest = hashlib.sha256(blob).hexdigest()
    packed = zlib.compress(blob, 6)
    # Several small attachments avoid notebook front-end limits for single outputs.
    parts = [packed[i:i+24576] for i in range(0,len(packed),24576)]
    for i, part in enumerate(parts, 1):
        token = f'SUDOKUSTRESS1.{name}.{i}.{len(parts)}.{digest}'
        href = 'data:application/octet-stream;base64,' + base64.b64encode(part).decode('ascii')
        display(HTML(f'<a download="{token}" href="{href}">{name} · {i}/{len(parts)}</a>'))
    return digest


def publish_json(name, value):
    raw = json.dumps(value, ensure_ascii=False, separators=(',', ':'), allow_nan=False).encode('utf-8')
    return publish_bytes(name, raw)


def recover_stress_files(notebook):
    """Read integrity-checked experiment records and weights from an executed .ipynb."""
    if isinstance(notebook, (str, Path)):
        notebook = json.loads(Path(notebook).read_text(encoding='utf-8-sig'))
    fragments = {}
    regex = re.compile(r'download="SUDOKUSTRESS1\.([a-zA-Z0-9_.-]+)\.(\d+)\.(\d+)\.([0-9a-f]{64})" href="data:application/octet-stream;base64,([a-zA-Z0-9+/=]+)"')
    for cell in notebook['cells']:
        for output in cell.get('outputs', []):
            html = ''.join(output.get('data', {}).get('text/html', ''))
            for name, index, total, digest, payload in regex.findall(html):
                index, total = int(index), int(total)
                if not 1 <= index <= total:
                    raise ValueError(f'Invalid part index for {name}')
                parts = fragments.setdefault((name,total,digest), {})
                chunk = base64.b64decode(payload, validate=True)
                if index in parts and parts[index] != chunk:
                    raise ValueError(f'Conflicting part {index} for {name}')
                parts[index] = chunk
    recovered = {}
    for (name,total,digest), parts in fragments.items():
        if len(parts) != total:
            continue
        raw = zlib.decompress(b''.join(parts[i] for i in range(1,total+1)))
        if hashlib.sha256(raw).hexdigest() != digest or (name in recovered and recovered[name] != raw):
            raise ValueError(f'Invalid or ambiguous archive member {name}')
        recovered[name] = raw
    return recovered


In [9]:
import hashlib
import json
import time
import threading
import traceback
from pathlib import Path


def stress_setup(rate_override, previous_cu):
    watchdog = None
    try:
        if not torch.cuda.is_available():
            raise RuntimeError('本实验需要带 CUDA 的 Colab GPU 运行时。')
        rate = float(rate_override)
        if rate == 0:
            watchdog = threading.Timer(180, release_colab_runtime)
            watchdog.daemon = True
            watchdog.start()
            rate = float(input('输入资源面板上当前会话的 CU/小时，再按回车：').strip())
        budget = RuntimeBudget(rate, previous_cu, session_start=STRESS_SESSION_START)
        budget.arm_watchdog()
        if watchdog:
            watchdog.cancel()
        budget.check()
        train_split = make_sudoku_splits(n_train=2048,n_val=256,n_test=256,seed=2026)['train']
        hash_train = hashlib.sha256(train_split[0].tobytes() + train_split[1].tobytes()).hexdigest()
        if hash_train != PRIOR_TRAIN_SHA:
            raise ValueError('训练数据哈希与第一轮正式实验不符。')
        cohorts = make_stress_sets()
        split_hashes = {k:hashlib.sha256(x.tobytes()+y.tobytes()).hexdigest() for k,(x,y) in cohorts.items()}
        if split_hashes != PRIOR_EVAL_SHA:
            raise ValueError('评估题或顺序与上轮不一致。')
        assert all(len(x)==512 and x.shape==y.shape==(512,81) for x,y in cohorts.values())
        protocol = dict(seeds=list(NEW_SEEDS), reused_seeds=list(REUSED_SEEDS),
                        analysis_seeds=list(REUSED_SEEDS+NEW_SEEDS), models=list(EXT_MODELS),
                        protocol_id=PROTOCOL_ID, prior_source_sha256=PRIOR_SOURCE_SHA, train_steps=5000,
                        train_loops=8, eval_loops=list(range(1,33)), batch=64, source_sha256=SOURCE_SHA,
                        training_sha256=hash_train, evaluation_sha256=split_hashes, data=stress_data_report(),
                        gpu=torch.cuda.get_device_name(), torch_version=str(torch.__version__),
                        primary='Hard cohort, split_gate minus full_gate, matched at R8; any loss by R32 and terminal repair at R32.',
                        secondary='Repeat on original cohort; report new seven seeds separately, any repair, and board accuracy.')
        where = Path('/content/sudoku_10seeds_weights')
        where.mkdir(exist_ok=True)
        publish_json('protocol.json',protocol)
        print('STRESS_READY',protocol['gpu'],rate,'CU/小时；原分布和难题各 512 道。',flush=True)
        return dict(splits={'train':train_split,**cohorts}, budget=budget, runs=[], metadata=protocol, weights=where)
    except BaseException:
        traceback.print_exc()
        release_colab_runtime()
        raise
    finally:
        if watchdog:
            watchdog.cancel()


def stress_run(ctx, variant, seed):
    try:
        budget = ctx['budget']
        budget.check()
        if variant not in EXT_MODELS or seed not in NEW_SEEDS:
            raise ValueError('本轮只运行两个门控模型和七个新增种子。')
        for existing in ctx['runs']:
            if (existing['variant'],existing['seed']) == (variant,seed):
                print(f'内核中已有完成记录：{variant}，种子 {seed}。', flush=True)
                return
        def progress(model, run_seed, step, total):
            print(f'STRESS_STEP {model} {run_seed} {step}/{total}', flush=True)
        print(f'STRESS_START {variant} {seed}', flush=True)
        rec = train_one_stress(variant,seed,ctx['splits'],steps=5000,weights_dir=ctx['weights'],
                               check_budget=budget.check,progress=progress)
        assert rec['steps']==5000 and set(rec['splits'])=={'original','hard'}
        weight_name = f'sudoku-{variant}-seed{seed}-steps5000.pt'
        blob = (ctx['weights']/weight_name).read_bytes()
        publish_json(f'run-{variant}-{seed}.json',rec)
        publish_bytes(weight_name,blob)
        ctx['runs'].append(rec)
        print(f'STRESS_SAVED {len(ctx["runs"])}/{len(EXT_MODELS)*len(NEW_SEEDS)} · {variant} · {seed} · 5000 步 · 32 轮逐格预测已保存', flush=True)
    except BaseException:
        print('STRESS_RUN_FAILED',variant,seed, flush=True)
        traceback.print_exc()
        time.sleep(2)
        release_colab_runtime()
        raise


def stress_finish(ctx):
    try:
        completed = [{k: r[k] for k in ('variant','seed','steps')} for r in ctx['runs']]
        expected = {(model, seed, 5000) for model in EXT_MODELS for seed in NEW_SEEDS}
        observed = {(r['variant'], r['seed'], r['steps']) for r in completed}
        if observed != expected or len(completed) != len(expected):
            raise ValueError('本轮新增运行清单不完整。')
        publish_json('extension_complete.json', dict(status='complete', completed=completed,
                     budget=ctx['budget'].summary(), protocol=ctx['metadata']))
        print(f'STRESS_COMPLETE {len(completed)}/{len(expected)} · 两组题的预测和十四份权重已保存。', flush=True)
    except BaseException:
        traceback.print_exc()
        raise
    finally:
        time.sleep(8)
        release_colab_runtime()
        ctx['budget'].disarm_watchdog()


In [10]:
# @title 已固定的两组题面、答案和独立求解器报告
STRESS_DATA_B64 = (
    '...'
    'IAEAAIITAQAAAA=='
)


In [11]:
# @title 初始化数据和计时
CU_PER_HOUR = 0.0  # @param {type:"number"}
PREVIOUS_CU = 0.0  # @param {type:"number"}
SOURCE_SHA = '6e89e4e4e73a36c31cff7721bdd72822df9c54d4d1ec05213e0146e61acef78b'
PRIOR_SOURCE_SHA = 'acbb251acdb41215f0b9712b3ad5481120035dac1dfbb9551a4fae2c21e14598'
PRIOR_EVAL_SHA = {'hard': '5ce4bdee0c641032873b0f3def0e0a94cf571505ffd343ead4f0afcd169a2ccf', 'original': '1c9c0a6a0a82e308ac04221550f7f730b907c618722bb51c341429ae7cddafb2'}
PROTOCOL_ID = 'sudoku-stress-ten-seed-extension-v1'
NEW_SEEDS = (59, 71, 83, 97, 109, 127, 139)
REUSED_SEEDS = (17, 29, 43)
EXT_MODELS = ('full_gate', 'split_gate')
PRIOR_TRAIN_SHA = '1c0486bb62b1bfb16d112ffc33b44c2f4ad2775149a84d49d28f41f768decd6b'
stress_context = stress_setup(CU_PER_HOUR, PREVIOUS_CU)


输入资源面板上当前会话的 CU/小时，再按回车：2.33


STRESS_READY NVIDIA L4 2.33 CU/小时；原分布和难题各 512 道。


In [12]:
# @title full_gate · 种子 59
stress_run(stress_context, 'full_gate', 59)


STRESS_START full_gate 59
STRESS_STEP full_gate 59 1000/5000
STRESS_STEP full_gate 59 2000/5000
STRESS_STEP full_gate 59 3000/5000
STRESS_STEP full_gate 59 4000/5000
STRESS_STEP full_gate 59 5000/5000


STRESS_SAVED 1/14 · full_gate · 59 · 5000 步 · 32 轮逐格预测已保存


In [13]:
# @title split_gate · 种子 59
stress_run(stress_context, 'split_gate', 59)


STRESS_START split_gate 59
STRESS_STEP split_gate 59 1000/5000
STRESS_STEP split_gate 59 2000/5000
STRESS_STEP split_gate 59 3000/5000
STRESS_STEP split_gate 59 4000/5000
STRESS_STEP split_gate 59 5000/5000


STRESS_SAVED 2/14 · split_gate · 59 · 5000 步 · 32 轮逐格预测已保存


In [14]:
# @title split_gate · 种子 71
stress_run(stress_context, 'split_gate', 71)


STRESS_START split_gate 71
STRESS_STEP split_gate 71 1000/5000
STRESS_STEP split_gate 71 2000/5000
STRESS_STEP split_gate 71 3000/5000
STRESS_STEP split_gate 71 4000/5000
STRESS_STEP split_gate 71 5000/5000


STRESS_SAVED 3/14 · split_gate · 71 · 5000 步 · 32 轮逐格预测已保存


In [15]:
# @title full_gate · 种子 71
stress_run(stress_context, 'full_gate', 71)


STRESS_START full_gate 71
STRESS_STEP full_gate 71 1000/5000
STRESS_STEP full_gate 71 2000/5000
STRESS_STEP full_gate 71 3000/5000
STRESS_STEP full_gate 71 4000/5000
STRESS_STEP full_gate 71 5000/5000


STRESS_SAVED 4/14 · full_gate · 71 · 5000 步 · 32 轮逐格预测已保存


In [16]:
# @title full_gate · 种子 83
stress_run(stress_context, 'full_gate', 83)


STRESS_START full_gate 83
STRESS_STEP full_gate 83 1000/5000
STRESS_STEP full_gate 83 2000/5000
STRESS_STEP full_gate 83 3000/5000
STRESS_STEP full_gate 83 4000/5000
STRESS_STEP full_gate 83 5000/5000


STRESS_SAVED 5/14 · full_gate · 83 · 5000 步 · 32 轮逐格预测已保存


In [17]:
# @title split_gate · 种子 83
stress_run(stress_context, 'split_gate', 83)


STRESS_START split_gate 83
STRESS_STEP split_gate 83 1000/5000
STRESS_STEP split_gate 83 2000/5000
STRESS_STEP split_gate 83 3000/5000
STRESS_STEP split_gate 83 4000/5000
STRESS_STEP split_gate 83 5000/5000


STRESS_SAVED 6/14 · split_gate · 83 · 5000 步 · 32 轮逐格预测已保存


In [18]:
# @title split_gate · 种子 97
stress_run(stress_context, 'split_gate', 97)


STRESS_START split_gate 97
STRESS_STEP split_gate 97 1000/5000
STRESS_STEP split_gate 97 2000/5000
STRESS_STEP split_gate 97 3000/5000
STRESS_STEP split_gate 97 4000/5000
STRESS_STEP split_gate 97 5000/5000


STRESS_SAVED 7/14 · split_gate · 97 · 5000 步 · 32 轮逐格预测已保存


In [19]:
# @title full_gate · 种子 97
stress_run(stress_context, 'full_gate', 97)


STRESS_START full_gate 97
STRESS_STEP full_gate 97 1000/5000
STRESS_STEP full_gate 97 2000/5000
STRESS_STEP full_gate 97 3000/5000
STRESS_STEP full_gate 97 4000/5000
STRESS_STEP full_gate 97 5000/5000


STRESS_SAVED 8/14 · full_gate · 97 · 5000 步 · 32 轮逐格预测已保存


In [20]:
# @title full_gate · 种子 109
stress_run(stress_context, 'full_gate', 109)


STRESS_START full_gate 109
STRESS_STEP full_gate 109 1000/5000
STRESS_STEP full_gate 109 2000/5000
STRESS_STEP full_gate 109 3000/5000
STRESS_STEP full_gate 109 4000/5000
STRESS_STEP full_gate 109 5000/5000


STRESS_SAVED 9/14 · full_gate · 109 · 5000 步 · 32 轮逐格预测已保存


In [21]:
# @title split_gate · 种子 109
stress_run(stress_context, 'split_gate', 109)


STRESS_START split_gate 109
STRESS_STEP split_gate 109 1000/5000
STRESS_STEP split_gate 109 2000/5000
STRESS_STEP split_gate 109 3000/5000
STRESS_STEP split_gate 109 4000/5000
STRESS_STEP split_gate 109 5000/5000


STRESS_SAVED 10/14 · split_gate · 109 · 5000 步 · 32 轮逐格预测已保存


In [22]:
# @title split_gate · 种子 127
stress_run(stress_context, 'split_gate', 127)


STRESS_START split_gate 127
STRESS_STEP split_gate 127 1000/5000
STRESS_STEP split_gate 127 2000/5000
STRESS_STEP split_gate 127 3000/5000
STRESS_STEP split_gate 127 4000/5000
STRESS_STEP split_gate 127 5000/5000


STRESS_SAVED 11/14 · split_gate · 127 · 5000 步 · 32 轮逐格预测已保存


In [23]:
# @title full_gate · 种子 127
stress_run(stress_context, 'full_gate', 127)


STRESS_START full_gate 127
STRESS_STEP full_gate 127 1000/5000
STRESS_STEP full_gate 127 2000/5000
STRESS_STEP full_gate 127 3000/5000
STRESS_STEP full_gate 127 4000/5000
STRESS_STEP full_gate 127 5000/5000


STRESS_SAVED 12/14 · full_gate · 127 · 5000 步 · 32 轮逐格预测已保存


In [24]:
# @title full_gate · 种子 139
stress_run(stress_context, 'full_gate', 139)


STRESS_START full_gate 139
STRESS_STEP full_gate 139 1000/5000
STRESS_STEP full_gate 139 2000/5000
STRESS_STEP full_gate 139 3000/5000
STRESS_STEP full_gate 139 4000/5000
STRESS_STEP full_gate 139 5000/5000


STRESS_SAVED 13/14 · full_gate · 139 · 5000 步 · 32 轮逐格预测已保存


In [25]:
# @title split_gate · 种子 139
stress_run(stress_context, 'split_gate', 139)


STRESS_START split_gate 139
STRESS_STEP split_gate 139 1000/5000
STRESS_STEP split_gate 139 2000/5000
STRESS_STEP split_gate 139 3000/5000
STRESS_STEP split_gate 139 4000/5000
STRESS_STEP split_gate 139 5000/5000


STRESS_SAVED 14/14 · split_gate · 139 · 5000 步 · 32 轮逐格预测已保存


In [ ]:
# @title 保存十四次新增运行清单并释放 GPU
stress_finish(stress_context)


STRESS_COMPLETE 14/14 · 两组题的预测和十四份权重已保存。
